In [ ]:
import pandas as pd
import json
from transformers import AutoTokenizer, pipeline
from src.download import download_and_split_dataset


In [ ]:
id_labels = {0: "anger", 1: "joy", 2: "optimism", 3: "sadness"}
train_df, val_df, test_df = download_and_split_dataset()
test_df['label_str'] = test_df['label'].map(id_labels)


In [ ]:
try:
    with open("error_analysis_indices.json", "r") as f:
        indices = json.load(f)
        both_wrong_zero = indices['both_wrong']
        bert_right_zero = indices['bert_right_rob_wrong']
        rob_right_zero = indices['rob_right_bert_wrong']
    print("Loaded zero-shot error indices.")
except FileNotFoundError:
    print("error_analysis_indices.json not found. Did you run the zero-shot notebook?")
    both_wrong_zero = []
    bert_right_zero = []
    rob_right_zero = []


In [ ]:
model_bert_id = "bdanko/bert-tweeteval-distilbert"
model_rob_id = "bdanko/bert-tweeteval-distilroberta"

pipe_bert = pipeline("text-classification", model=model_bert_id)
pipe_rob = pipeline("text-classification", model=model_rob_id)

tokenizer_bert = AutoTokenizer.from_pretrained(model_bert_id)
tokenizer_rob = AutoTokenizer.from_pretrained(model_rob_id)


In [ ]:
def predict_batch(texts, pipe):
    preds = pipe(texts, truncation=True, max_length=512)
    return [p['label'].lower() for p in preds]

print("Generating predictions...")
texts = test_df['text'].tolist()
test_df['distilbert_pred'] = predict_batch(texts, pipe_bert)
test_df['distilroberta_pred'] = predict_batch(texts, pipe_rob)
print("Done!")


## 1. Performance on Previous Zero-Shot Misclassifications
Checking how the fine-tuned models behave on the previously problematic samples.

In [ ]:
def print_sample_analysis(indices, title):
    if not indices:
        return
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    
    subset = test_df.loc[indices]
    for idx, row in subset.iterrows():
        text = row['text']
        true_label = row['label_str']
        pred_bert = row['distilbert_pred']
        pred_rob = row['distilroberta_pred']
        
        tokens_bert = tokenizer_bert.tokenize(text)
        tokens_rob = tokenizer_rob.tokenize(text)
        
        print(f"Text: {text}")
        print(f"True: {true_label} | Fine-Tuned DistilBERT Pred: {pred_bert} | Fine-Tuned DistilRoBERTa Pred: {pred_rob}")
        print(f"WordPiece (DistilBERT): {tokens_bert}")
        print(f"BPE (DistilRoBERTa): {tokens_rob}")
        print("-" * 50)

print_sample_analysis(both_wrong_zero, "Samples BOTH zero-shot models misclassified")
print_sample_analysis(bert_right_zero, "Samples zero-shot DistilBERT got right, but DistilRoBERTa missed")
print_sample_analysis(rob_right_zero, "Samples zero-shot DistilRoBERTa got right, but DistilBERT missed")


## 2. New Fine-Tuned Model Errors
Finding samples that specifically cause issues for our fine-tuned models.

In [ ]:
both_wrong_ft = test_df[(test_df['distilbert_pred'] != test_df['label_str']) & (test_df['distilroberta_pred'] != test_df['label_str'])].head(8)
bert_right_rob_wrong_ft = test_df[(test_df['distilbert_pred'] == test_df['label_str']) & (test_df['distilroberta_pred'] != test_df['label_str'])].head(8)
rob_right_bert_wrong_ft = test_df[(test_df['distilroberta_pred'] == test_df['label_str']) & (test_df['distilbert_pred'] != test_df['label_str'])].head(8)

print_sample_analysis(both_wrong_ft.index.tolist(), "NEW: Misclassified by BOTH Fine-Tuned Models")
print_sample_analysis(bert_right_rob_wrong_ft.index.tolist(), "NEW: Fine-Tuned DistilBERT Right, DistilRoBERTa Wrong")
print_sample_analysis(rob_right_bert_wrong_ft.index.tolist(), "NEW: Fine-Tuned DistilRoBERTa Right, DistilBERT Wrong")
